# px_high Analysis — DoubleML PLR
**Ashley Thompson — Capstone**

Replicates ml_analysis with `px_high` as the outcome variable.
If negative sentiment drives `px_high` down, it confirms sentiment suppresses
the intraday price ceiling, not just the open-to-close return.

- **Run 1:** Twitter sentiment as treatment, `px_high` as outcome
- **Run 2:** Negative tweet count as treatment, `px_high` as outcome (Teti et al. 2019)

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Install dependencies

In [ ]:
!pip install -q doubleml xgboost scikit-learn

## 2. Load data
Run Option A or Option B, then skip the other.

In [ ]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

In [ ]:
# Option B — Manual upload (data only)
from google.colab import files
uploaded = files.upload()
DATA_PATH = 'panel_long.csv'

In [ ]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor

np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')
long.head()

In [ ]:
def make_xgb():
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        device           = 'cuda',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    )

## 4. Run 1 — Twitter sentiment → px_high
**Treatment:** `twitter_sent` | **Outcome:** `px_high`  
`return` excluded from confounders (derived from `px_open`/`px_close`, not `px_high`).

In [ ]:
confounders_1 = [
    'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_count', 'news_sent', 'rsi_30', 'ma_50',
    'twitter_neg_count', 'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

df1 = (
    long[['px_high', 'twitter_sent'] + confounders_1]
    .dropna()
    .reset_index(drop=True)
)

dml_1 = dml.DoubleMLPLR(
    dml.DoubleMLData(df1, y_col='px_high', d_cols='twitter_sent', x_cols=confounders_1),
    ml_l    = make_xgb(),
    ml_m    = make_xgb(),
    n_folds = 5,
    n_rep   = 20,
)
dml_1.fit()
dml_1.bootstrap(method='normal', n_rep_boot=1000)

print(f'N (complete cases): {len(df1):,}')
print(dml_1.summary)
print(dml_1.confint(level=0.95))

## 5. Run 2 — Negative tweet count → px_high (Teti et al. 2019)
**Treatment:** `twitter_neg_count` | **Outcome:** `px_high`  
Both `twitter_count` and `twitter_sent` stay in confounders to control for total volume and sentiment direction independently.

In [ ]:
confounders_2 = [
    'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_sent', 'twitter_count', 'news_sent', 'rsi_30', 'ma_50',
    'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

df2 = (
    long[['px_high', 'twitter_neg_count'] + confounders_2]
    .dropna()
    .reset_index(drop=True)
)

dml_2 = dml.DoubleMLPLR(
    dml.DoubleMLData(df2, y_col='px_high', d_cols='twitter_neg_count', x_cols=confounders_2),
    ml_l    = make_xgb(),
    ml_m    = make_xgb(),
    n_folds = 5,
    n_rep   = 20,
)
dml_2.fit()
dml_2.bootstrap(method='normal', n_rep_boot=1000)

print(f'N (complete cases): {len(df2):,}')
print(dml_2.summary)
print(dml_2.confint(level=0.95))

## 6. Export — Publication-ready LaTeX table
Saves `dml_px_high_results.tex` to your output folder with both runs side by side.

In [ ]:
def dml_to_latex(models, labels, title, notes=''):
    """Build a publication-ready LaTeX table from DoubleMLPLR models."""
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.1:  return '*'
        return ''

    all_treatments = []
    for m in models:
        for t in m.summary.index:
            if t not in all_treatments:
                all_treatments.append(t)

    ncols = len(models)
    col_fmt = 'l' + 'c' * ncols

    lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        f'\\caption{{{title}}}',
        f'\\begin{{tabular}}{{{col_fmt}}}',
        r'\hline\hline',
        ' & ' + ' & '.join([f'({i+1}) {labels[i]}' for i in range(ncols)]) + r' \\',
        r'\hline',
    ]

    for t in all_treatments:
        t_label = t.replace('_', r'\_')
        coef_cells, se_cells = [], []
        for m in models:
            if t in m.summary.index:
                coef = m.summary.loc[t, 'coef']
                se   = m.summary.loc[t, 'std err']
                pval = m.summary.loc[t, 'P>|t|']
                coef_cells.append(f'{coef:.6f}{stars(pval)}')
                se_cells.append(f'({se:.6f})')
            else:
                coef_cells.append('---')
                se_cells.append('')
        lines.append(t_label + ' & ' + ' & '.join(coef_cells) + r' \\')
        lines.append('& ' + ' & '.join(se_cells) + r' \\')

    lines += [
        r'\hline',
        'Estimator & ' + ' & '.join(['DoubleML PLR'] * ncols) + r' \\',
        'ML Learner & ' + ' & '.join(['XGBoost (GPU)'] * ncols) + r' \\',
        r'\hline\hline',
    ]
    if notes:
        lines.append(
            f'\\multicolumn{{{ncols + 1}}}{{l}}'
            f'{{\\footnotesize \\textit{{Notes:}} {notes}}} \\\\'
        )
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

In [ ]:
tex = dml_to_latex(
    [dml_1, dml_2],
    ['Twitter Sentiment', 'Neg Tweet Count'],
    title='DoubleML PLR --- Sentiment and Intraday Price High (px\\_high)',
    notes=(
        'Heteroskedasticity-robust SE in parentheses. '
        'XGBoost learner with 5-fold cross-fitting, 20 repetitions, 1000 bootstrap draws. '
        'Outcome is intraday high price (px\\_high); return excluded from confounders. '
        'Run 2 per Teti et al. (2019). '
        '*** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
    ),
)
with open(OUTPUT_PATH + 'dml_px_high_results.tex', 'w') as f:
    f.write(tex)
print('Saved dml_px_high_results.tex')
print(tex)

In [ ]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])

staged   = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(
    ['git', 'log', 'origin/main..HEAD', '--oneline'],
    capture_output=True, text=True
)

if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add px_high analysis results from Colab'], check=True)

if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit or push.')